# Neural Speaker ID Pipeline — Log-Mel Spectrogram → CNN

**Goal:** Treat each 3-second clip as a 64×300 log-mel spectrogram image, train a small 3-layer CNN classifier, plot training curves + confusion matrix, and save the model.

Uses CUDA (RTX 4060) if available, falls back to CPU.

**Before running:**  
Same data layout as notebook 01 — `data/recordings/<speaker_name>/*.wav`

## 1. Imports and configuration

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from features.spectrogram import extract_logmel
from models.cnn import SpeakerCNN

DATA_DIR   = ROOT / 'data' / 'recordings'
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RANDOM_STATE = 42
BATCH_SIZE   = 32
EPOCHS       = 40
LR           = 1e-3

print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU:    {torch.cuda.get_device_name(0)}')

## 2. Load dataset

We extract a (64, 300) log-mel spectrogram for every clip and store them in RAM. For 4 speakers × 60 clips ≈ 240 spectrograms this is fine (~55 MB).

In [ ]:
speaker_dirs = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])

if not speaker_dirs:
    raise RuntimeError(
        f'No speaker directories found in {DATA_DIR}.\n'
        'Run: python scripts/generate_placeholder_data.py'
    )

X_list, y_list = [], []

for spk_dir in speaker_dirs:
    wavs = sorted(spk_dir.glob('*.wav'))
    print(f'  {spk_dir.name}: {len(wavs)} clips', end=' ... ', flush=True)
    for wav in wavs:
        try:
            mel = extract_logmel(str(wav))   # (64, 300)
            X_list.append(mel)
            y_list.append(spk_dir.name)
        except Exception as e:
            print(f'\n    skip {wav.name}: {e}')
    print('ok')

X = np.stack(X_list)  # (N, 64, 300)
y = np.array(y_list)

le = LabelEncoder()
y_enc = le.fit_transform(y)

print(f'\nDataset: {X.shape}  Classes: {list(le.classes_)}')

## 3. Split 80 / 10 / 10 and build DataLoaders

In [ ]:
class SpectrogramDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        # Add channel dim: (N, 1, 64, 300)
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


idx_all = np.arange(len(X))
idx_tv, idx_test = train_test_split(idx_all, test_size=0.10, stratify=y_enc, random_state=RANDOM_STATE)
idx_train, idx_val = train_test_split(idx_tv,   test_size=0.111, stratify=y_enc[idx_tv], random_state=RANDOM_STATE)

train_ds = SpectrogramDataset(X[idx_train], y_enc[idx_train])
val_ds   = SpectrogramDataset(X[idx_val],   y_enc[idx_val])
test_ds  = SpectrogramDataset(X[idx_test],  y_enc[idx_test])

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')

## 4. Model architecture

Three convolutional blocks followed by adaptive average pooling and a two-layer MLP head.  
Input shape: **(batch, 1, 64, 300)** — one channel, 64 mel bins, 300 time steps.

In [ ]:
n_classes = len(le.classes_)
model = SpeakerCNN(n_classes=n_classes).to(DEVICE)
print(model)

# Sanity-check forward pass
dummy = torch.zeros(2, 1, 64, 300).to(DEVICE)
out   = model(dummy)
print(f'\nOutput shape: {out.shape}  (expected: [2, {n_classes}])')

## 5. Training loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

history = {'train_loss': [], 'val_loss': [], 'val_acc': []}


def run_epoch(dl, train=True):
    model.train(train)
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for xb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if train:
                optimizer.zero_grad()
            out  = model(xb)
            loss = criterion(out, yb)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(yb)
            correct    += (out.argmax(1) == yb).sum().item()
            total      += len(yb)
    return total_loss / total, correct / total


best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    tr_loss, _         = run_epoch(train_dl, train=True)
    val_loss, val_acc  = run_epoch(val_dl,   train=False)
    scheduler.step(val_loss)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODELS_DIR / 'cnn.pt')

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS}  '
              f'train_loss={tr_loss:.4f}  '
              f'val_loss={val_loss:.4f}  '
              f'val_acc={val_acc:.3f}  '
              f'{"★" if val_acc == best_val_acc else ""}')

print(f'\nBest val accuracy: {best_val_acc:.3f}')

## 6. Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5), facecolor='#111')

ep = range(1, EPOCHS + 1)
ax1.plot(ep, history['train_loss'], color='#4af', label='train')
ax1.plot(ep, history['val_loss'],   color='#f84', label='val')
ax1.set_title('Loss', color='#fff'); ax1.set_facecolor('#111')
ax1.tick_params(colors='#888'); ax1.legend(labelcolor='#aaa')

ax2.plot(ep, [a * 100 for a in history['val_acc']], color='#4d4')
ax2.set_title('Val Accuracy (%)', color='#fff'); ax2.set_facecolor('#111')
ax2.tick_params(colors='#888')

plt.tight_layout()
plt.savefig(MODELS_DIR / 'cnn_training.png', dpi=120, facecolor='#111')
plt.show()

## 7. Test set evaluation

In [ ]:
# Reload best checkpoint
model.load_state_dict(torch.load(MODELS_DIR / 'cnn.pt', map_location=DEVICE))

all_preds, all_true = [], []
model.eval()
with torch.no_grad():
    for xb, yb in test_dl:
        preds = model(xb.to(DEVICE)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(yb.numpy())

print('=== Test set ===')
print(classification_report(all_true, all_preds, target_names=le.classes_))

## 8. Confusion matrix

In [ ]:
cm = confusion_matrix(all_true, all_preds)
fig, ax = plt.subplots(figsize=(5, 4), facecolor='#111')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_facecolor('#111')
ax.tick_params(colors='#aaa')
ax.xaxis.label.set_color('#aaa')
ax.yaxis.label.set_color('#aaa')
ax.title.set_color('#fff')
ax.set_title('CNN — Test Confusion Matrix')
plt.tight_layout()
plt.savefig(MODELS_DIR / 'cnn_confusion.png', dpi=120, facecolor='#111')
plt.show()

## 9. Save class list

The model weights were already saved as the best checkpoint during training.  
`scripts/serve.py` reads `models/cnn.pt` and `models/cnn_classes.json`.

In [ ]:
(MODELS_DIR / 'cnn_classes.json').write_text(json.dumps(list(le.classes_)))

print('Saved:')
print('  models/cnn.pt')
print(f'  models/cnn_classes.json  →  {list(le.classes_)}')
print('\nRe-start the server to pick up the new model.')